# Step 04.1 — Linear：从矩阵乘法到可学习层

这一节建立一个核心认识：

> `nn.Linear` 的数学核心仍然只是矩阵乘法。

但在进入 `nn.Linear` 前，我们先把一个很容易混淆的问题讲清楚：**`(2,)`、`(1,2)`、`(2,1)` 到底有什么区别，以及一维向量参与 `@` 时发生了什么。**

## 1. `(2,)` 不是 `(1,2)`：先区分 vector、row matrix、column matrix

先准备同样的两个数字 `[2, -1]`，但用三种 shape 表示。

In [ ]:
import mlx.core as mx

x = mx.array([2.0, -1.0])
x_row = x.reshape(1, 2)
x_col = x.reshape(2, 1)

print("x     =", x, "shape:", x.shape)
print("x_row =", x_row, "shape:", x_row.shape)
print("x_col =")
print(x_col)
print("shape:", x_col.shape)

三者虽然都含有两个数，但结构不同：

```text
x.shape = (2,)
[ 2, -1 ]
```

这是 **1D vector**。它只有一个 axis，长度为 2；它本身没有“1 行 2 列”这个二维结构。

```text
x_row.shape = (1, 2)
[[ 2, -1 ]]
```

这是 **row matrix**：1 行 2 列。

```text
x_col.shape = (2, 1)
[[ 2],
 [-1]]
```

这是 **column matrix**：2 行 1 列。

所以 `(2,)` 并不等于 `(1,2)`，只是某些矩阵乘法规则会在运算内部临时把 1D vector 按行或按列解释。

## 2. 为什么权重 `W` 是 `(3,2)`？

输入有 2 个特征：

`x = [x₀, x₁]`

现在希望产生 3 个输出特征：

`y = [y₀, y₁, y₂]`

每一个输出都要由输入的两个特征加权得到：

```text
y₀ = w₀₀ x₀ + w₀₁ x₁
y₁ = w₁₀ x₀ + w₁₁ x₁
y₂ = w₂₀ x₀ + w₂₁ x₁
```

因此需要 **3 组权重**，每组 **2 个数**：

```text
W =
[[w₀₀, w₀₁],   <- 负责产生 y₀
 [w₁₀, w₁₁],   <- 负责产生 y₁
 [w₂₀, w₂₁]]   <- 负责产生 y₂

W.shape = (3, 2)
```

也就是：

**输出特征数 × 输入特征数 = `(3,2)`**。

In [ ]:
W = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])

print("W =")
print(W)
print("W shape  :", W.shape)
print("W.T shape:", W.T.shape)

## 3. 两种完全等价的矩阵写法：列向量写法 vs MLX 常用的最后一维特征写法

如果你在线性代数课里把向量写成**列向量**，最自然的是：

```text
W      @ x_col  -> y_col
(3,2)   (2,1)     (3,1)
```

这是标准的 `y = W x`。

而深度学习框架通常把 feature 放在 tensor 的**最后一个 axis**。单个样本若显式写成二维 row matrix，就是：

```text
x_row  @ W.T   -> y_row
(1,2)   (2,3)    (1,3)
```

两种写法只是数据摆放方向不同，算出的三个数字相同。

In [ ]:
y_col = W @ x_col
y_row = x_row @ W.T

print("column-vector form:")
print(y_col)
print("shape:", y_col.shape)

print("\nrow-vector form:")
print(y_row)
print("shape:", y_row.shape)

手算得到的三个输出是：

- `y₀ = 2×1 + (-1)×0 = 2`
- `y₁ = 2×0 + (-1)×1 = -1`
- `y₂ = 2×1 + (-1)×1 = 1`

所以无论写成 `(3,1)` 的列矩阵还是 `(1,3)` 的行矩阵，本质上都保存 `[2, -1, 1]` 这三个输出特征。

## 4. 那为什么 `x.shape == (2,)` 也能直接写 `x @ W.T`？

现在回到最容易误解的代码：

```python
y = x @ W.T
```

此时：

```text
x.shape   = (2,)
W.T.shape = (2,3)
```

严格来说，`x` 仍然是一维 vector，并没有在变量本身上变成 `(1,2)`。

矩阵乘法 `@` 对 **1D 左操作数**有一个特殊规则：为了完成矩阵乘法，可以把它在计算语义上暂时看成：

```text
(2,)  ->  (1,2)
```

于是内部可以理解为：

```text
(1,2) @ (2,3) -> (1,3)
```

计算完成后，刚才为了乘法临时补进去的那个最前面的 `1` 又被去掉，因此真正返回：

```text
(3,)
```

所以要特别区分：

- `x.shape` 从头到尾仍然是 `(2,)`；
- `(1,2)` 是理解 `1D @ 2D` 矩阵乘法时的**临时计算语义**；
- 返回结果会把那个临时维度移除。

In [ ]:
y_1d = x @ W.T
y_explicit = x.reshape(1, 2) @ W.T

print("x @ W.T:")
print(y_1d)
print("shape:", y_1d.shape)

print("\nexplicit (1,2) @ (2,3):")
print(y_explicit)
print("shape:", y_explicit.shape)

print("\nx is still:", x.shape)

你应该看到：

```text
x @ W.T                  -> shape (3,)
x.reshape(1,2) @ W.T     -> shape (1,3)
x.shape                  -> 仍然是 (2,)
```

三个输出数字相同，只是结果的 rank 不同。

## 5. 为什么深度学习里很快就不需要纠结这个特殊规则？

真实 MiniGPT 的 hidden state 通常不是 `(C,)`，而是：

```text
(B, T, C)
```

例如 `(2,4,128)`：2 个样本、每个 4 个 token、每个 token 128 个特征。

Linear 只把**最后一个特征维**从 `C_in` 映射到 `C_out`，前面的维度保持不变：

```text
(B, T, C_in) -> Linear -> (B, T, C_out)
```

所以 `(2,)` 的特殊情况主要是我们现在为了手算而故意使用的极小例子。

In [ ]:
X = mx.array([
    [2.0, -1.0],
    [3.0,  4.0],
])

Y = X @ W.T

print("X shape  :", X.shape)
print("W.T shape:", W.T.shape)
print("Y shape  :", Y.shape)
print(Y)

这里没有 1D 特殊情况：

```text
(2,2) @ (2,3) -> (2,3)
```

每一行输入都使用同一套 `W`。

## 6. 回到 MLX 的 `nn.Linear`

`mlx.nn` 是 MLX 的 neural-network module library。

`nn.Linear(2, 3)` 表示：把输入 tensor 的**最后一个维度**从 2 映射到 3。

MLX 把权重保存成：

```text
weight.shape = (output_dims, input_dims) = (3,2)
```

因此其核心计算对应：

```python
x @ weight.T
```


### `layer` 到底是什么？为什么它看起来像函数？

执行：

```python
layer = nn.Linear(2, 3, bias=False)
```

右边的 `nn.Linear(...)` 会**创建一个 Linear 对象**，然后把这个对象保存到变量 `layer`。

所以 `layer` 不是普通函数，而是一个 **对象（object）**，它里面保存了这一层自己的状态，例如：

- `layer.weight`：权重矩阵；
- 如果启用 bias，还会有 `layer.bias`；
- 以及“收到输入时应该怎样计算输出”的前向计算逻辑。

可以先把它想成一个小机器：

```text
layer
├── weight
├── bias（可选）
└── forward rule
```

Python 允许某些对象定义 `__call__` 行为，所以对象也能像函数一样写成：

```python
out = layer(x)
```

这并不表示 `layer` 变成了普通函数，而是表示：**调用这个对象的前向计算逻辑**。

对 `nn.Linear` 来说，忽略 bias 时，概念上等价于：

```python
out = x @ layer.weight.T
```

所以：

```python
layer = nn.Linear(...)
```

是在**创建层**；而：

```python
layer(x)
```

是在**让这个层处理输入**。

In [ ]:
import mlx.nn as nn

layer = nn.Linear(2, 3, bias=False)

print("Linear weight shape:", layer.weight.shape)

out_1d = layer(x)
out_batch = layer(X)
mx.eval(out_1d, out_batch)

print("1D input :", x.shape, "->", out_1d.shape)
print("2D input :", X.shape, "->", out_batch.shape)

现在权重还是随机初始化，因此具体输出数字暂时没有意义。

### 本节结论

- `(2,)` 是 1D vector，不是 `(1,2)`；
- `(1,2)` 是 1×2 row matrix，`(2,1)` 是 2×1 column matrix；
- 对 `1D @ 2D`，`@` 会使用特殊的 1D matmul 规则，可以把它理解成临时补一维完成矩阵乘法，再移除该临时维度；
- MLX 的 Linear 权重按 `(output_dims, input_dims)` 保存；
- `Linear(2,3)` 的本质是用 3 组、每组 2 个权重，把 2 个输入特征混合成 3 个输出特征；
- 真正 MiniGPT 中输入通常是 `(B,T,C)`，Linear 主要变换最后一个特征维。